# ROGII Wellbore Geology

## Score: 41.687

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
SAMPLE_SUB = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv")

ROLL_WINDOWS = (5, 15, 31)
DTW_STEP = 3
DTW_BAND = 45
SLOPE_LOOKBACK = 50
LGB_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100
N_FOLDS = 5

In [ ]:
def list_wells(data_dir: Path) -> list[str]:
    wells = sorted({p.name.split("__")[0] for p in data_dir.glob("*__horizontal_well.csv")})
    return wells


def load_horizontal(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__horizontal_well.csv"
    df = pd.read_csv(path)
    df.insert(0, "well", well)
    df["row_idx"] = np.arange(len(df))
    return df


def load_typewell(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__typewell.csv"
    tw = pd.read_csv(path).sort_values("TVT").reset_index(drop=True)
    return tw


def interp_typewell_gr(tvt_values: np.ndarray, tw: pd.DataFrame) -> np.ndarray:
    tvt = tw["TVT"].to_numpy(dtype=float)
    gr = tw["GR"].to_numpy(dtype=float)
    return np.interp(tvt_values, tvt, gr, left=gr[0], right=gr[-1])


def eval_start_index(df: pd.DataFrame) -> int:
    need_pred = df["TVT_input"].isna()
    if need_pred.any():
        return int(need_pred.idxmax())
    if df["GR"].notna().any():
        return int(df["GR"].notna().idxmax())
    return len(df)


def pre_eval_slope(h: pd.DataFrame, eval_start: int) -> float:
    i0 = max(0, eval_start - SLOPE_LOOKBACK)
    seg = h.iloc[i0:eval_start]
    if len(seg) < 5:
        return 1.0
    if "TVT" in seg.columns and seg["TVT"].notna().sum() >= 5:
        tvt = seg["TVT"].astype(float)
    else:
        tvt = seg["TVT_input"].astype(float)
    md = seg["MD"].astype(float)
    valid = tvt.notna() & md.notna()
    tvt = tvt[valid]
    md = md[valid]
    if len(tvt) < 5:
        return 1.0
    d_md = md.iloc[-1] - md.iloc[0]
    if abs(d_md) < 1e-6:
        return 1.0
    return float(np.clip((tvt.iloc[-1] - tvt.iloc[0]) / d_md, 0.05, 3.0))


def zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    s = x.std()
    if s < 1e-6:
        return np.zeros_like(x)
    return (x - x.mean()) / s


def dtw_band_path(x: np.ndarray, y: np.ndarray, band: int) -> tuple[np.ndarray, np.ndarray]:
    x = zscore(x)
    y = zscore(y)
    n, m = len(x), len(y)
    inf = 1e18
    d = np.full((n + 1, m + 1), inf)
    d[0, 0] = 0.0
    for i in range(1, n + 1):
        j_lo = max(1, i - band)
        j_hi = min(m, i + band)
        for j in range(j_lo, j_hi + 1):
            cost = (x[i - 1] - y[j - 1]) ** 2
            d[i, j] = cost + min(d[i - 1, j], d[i, j - 1], d[i - 1, j - 1])
    i, j = n, m
    path_i, path_j = [], []
    while i > 0 and j > 0:
        path_i.append(i - 1)
        path_j.append(j - 1)
        step = np.argmin([d[i - 1, j], d[i, j - 1], d[i - 1, j - 1]])
        if step == 0:
            i -= 1
        elif step == 1:
            j -= 1
        else:
            i -= 1
            j -= 1
    return np.array(path_i[::-1]), np.array(path_j[::-1])


def compute_tvt_dtw(h: pd.DataFrame, tw: pd.DataFrame, eval_start: int) -> np.ndarray:
    n = len(h)
    out = np.full(n, np.nan, dtype=float)
    gr = h["GR"].iloc[eval_start:].astype(float).ffill().bfill().to_numpy()
    if len(gr) < 10:
        return out

    if eval_start > 0:
        if pd.notna(h["TVT_input"].iloc[eval_start - 1]):
            anchor_tvt = float(h["TVT_input"].iloc[eval_start - 1])
        elif "TVT" in h.columns and pd.notna(h["TVT"].iloc[eval_start - 1]):
            anchor_tvt = float(h["TVT"].iloc[eval_start - 1])
        else:
            anchor_tvt = float(h["MD"].iloc[eval_start - 1])
    else:
        anchor_tvt = float(h["TVT_input"].iloc[0]) if pd.notna(h["TVT_input"].iloc[0]) else float(h["MD"].iloc[0])

    slope = pre_eval_slope(h, eval_start)
    tw_tvt = tw["TVT"].to_numpy(dtype=float)
    tw_gr = tw["GR"].to_numpy(dtype=float)
    lo = anchor_tvt - 40.0
    hi = anchor_tvt + max(len(gr) * slope * 1.25 + 80.0, 200.0)
    mask = (tw_tvt >= lo) & (tw_tvt <= hi)
    ref_tvt = tw_tvt[mask]
    ref_gr = tw_gr[mask]
    if len(ref_gr) < 20:
        out[eval_start : eval_start + len(gr)] = anchor_tvt + np.arange(len(gr)) * slope
        return out

    gr_ds = gr[::DTW_STEP]
    ref_n = max(int(len(gr_ds) * 1.4), len(gr_ds) + 10)
    ref_tvt_u = np.linspace(ref_tvt[0], ref_tvt[-1], ref_n)
    ref_gr_u = np.interp(ref_tvt_u, ref_tvt, ref_gr)

    path_i, path_j = dtw_band_path(gr_ds, ref_gr_u, DTW_BAND)
    tvt_eval = np.empty(len(gr), dtype=float)
    for k in range(len(gr)):
        i_target = min(k // DTW_STEP, len(path_i) - 1)
        idx = int(np.argmin(np.abs(path_i - i_target)))
        tvt_eval[k] = ref_tvt_u[path_j[idx]]

    out[eval_start : eval_start + len(gr)] = tvt_eval
    return out


def build_features(h: pd.DataFrame, tw: pd.DataFrame, simulate_eval: bool = False) -> pd.DataFrame:
    df = h.copy()
    eval_start = eval_start_index(df)
    df["eval_start"] = eval_start
    df["in_eval"] = df["row_idx"] >= eval_start

    tvt_in = df["TVT_input"].copy()
    if simulate_eval and not tvt_in.isna().all():
        tvt_in.iloc[eval_start:] = np.nan

    prev_tvt = tvt_in.shift(1)
    df["tvt_anchor"] = prev_tvt.ffill()
    anchor_md = df["MD"].shift(1).where(prev_tvt.notna()).ffill()
    df["md_anchor"] = anchor_md
    df["md_since_anchor"] = df["MD"] - df["md_anchor"]
    df["tvt_linear"] = df["tvt_anchor"] + df["md_since_anchor"]
    df["ft_into_eval"] = np.maximum(0, df["row_idx"] - eval_start)

    df["d_md"] = df["MD"].diff().fillna(0)
    df["d_z"] = df["Z"].diff().fillna(0)

    gr = df["GR"].astype(float)
    gr_filled = gr.ffill()
    for w in ROLL_WINDOWS:
        df[f"gr_roll_mean_{w}"] = gr_filled.rolling(w, min_periods=1).mean()
        df[f"gr_roll_std_{w}"] = gr_filled.rolling(w, min_periods=1).std().fillna(0)
    df["gr_missing"] = gr.isna().astype(np.int8)

    tvt_dtw = compute_tvt_dtw(h, tw, eval_start)
    df["tvt_dtw"] = np.where(np.isnan(tvt_dtw), df["tvt_linear"], tvt_dtw)
    df["tvt_dtw_minus_linear"] = df["tvt_dtw"] - df["tvt_linear"]

    tvt_lin = df["tvt_linear"].to_numpy(dtype=float)
    df["tw_gr_at_linear"] = interp_typewell_gr(tvt_lin, tw)
    df["tw_gr_at_dtw"] = interp_typewell_gr(df["tvt_dtw"].to_numpy(dtype=float), tw)
    df["gr_minus_tw"] = gr_filled - df["tw_gr_at_linear"]
    df["gr_minus_tw_dtw"] = gr_filled - df["tw_gr_at_dtw"]

    if "TVT" in df.columns:
        df["delta_tvt"] = df["TVT"] - df.groupby("well")["TVT"].shift(1)

    return df


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "d_md", "d_z",
    "tvt_anchor", "md_since_anchor", "tvt_linear", "tvt_dtw", "tvt_dtw_minus_linear",
    "ft_into_eval",
    "GR", "gr_missing",
    "tw_gr_at_linear", "tw_gr_at_dtw", "gr_minus_tw", "gr_minus_tw_dtw",
    *[f"gr_roll_mean_{w}" for w in ROLL_WINDOWS],
    *[f"gr_roll_std_{w}" for w in ROLL_WINDOWS],
]


def featurize_well(well: str, data_dir: Path, simulate_eval: bool = False) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    return build_features(h, tw, simulate_eval=simulate_eval)


def deltas_to_tvt(sub: pd.DataFrame, delta_pred: np.ndarray) -> np.ndarray:
    sub = sub.sort_values("row_idx")
    anchor = sub["tvt_anchor"].iloc[0]
    return anchor + np.cumsum(delta_pred)


def calibrate_well_tvt(sub: pd.DataFrame, tvt: np.ndarray) -> np.ndarray:
    sub = sub.sort_values("row_idx")
    anchor = float(sub["tvt_anchor"].iloc[0])
    tvt = np.asarray(tvt, dtype=float).copy()
    tvt += anchor - tvt[0]
    dtw0 = sub["tvt_dtw"].iloc[0]
    if np.isfinite(dtw0):
        tvt += 0.25 * (anchor - dtw0)
    return tvt

In [ ]:
def build_training_frame(wells: list[str], data_dir: Path) -> pd.DataFrame:
    parts = []
    for well in wells:
        df = featurize_well(well, data_dir, simulate_eval=True)
        if "TVT" not in df.columns:
            continue
        mask = (
            df["in_eval"]
            & df["GR"].notna()
            & df["TVT"].notna()
            & df["tvt_anchor"].notna()
            & df["delta_tvt"].notna()
        )
        parts.append(df.loc[mask])
    return pd.concat(parts, ignore_index=True)


train_wells = list_wells(TRAIN_DIR)
print(f"Training wells: {len(train_wells)}")

train_df = build_training_frame(train_wells, TRAIN_DIR)
print(f"Training rows (eval zone only): {len(train_df):,}")
print(train_df[["TVT", "delta_tvt", "tvt_linear", "GR"]].describe().round(3))

In [ ]:
X = train_df[FEATURE_COLS]
y = train_df["delta_tvt"].astype(float)
groups = train_df["well"]

oof_tvt = np.zeros(len(train_df))
models = []

gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups)):
    dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
    )
    models.append(model)
    va = train_df.iloc[va_idx]
    delta_pred = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)

    for well in va["well"].unique():
        well_mask = (va["well"] == well).to_numpy()
        wpos = va_idx[well_mask]
        oof_tvt[wpos] = calibrate_well_tvt(train_df.iloc[wpos], deltas_to_tvt(train_df.iloc[wpos], delta_pred[well_mask]))

    fold_rmse = np.sqrt(mean_squared_error(train_df.iloc[va_idx]["TVT"], oof_tvt[va_idx]))
    print(f"Fold {fold + 1}/{N_FOLDS}  eval-zone TVT RMSE = {fold_rmse:.4f}  (best_iter={model.best_iteration})")

cv_rmse = np.sqrt(mean_squared_error(train_df["TVT"], oof_tvt))
print(f"\nOOF eval-zone TVT RMSE (GroupKFold by well): {cv_rmse:.4f}")

In [ ]:
def predict_eval_zone(well: str, data_dir: Path, models: list) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    df = build_features(h, tw, simulate_eval=False)
    sub = df.loc[df["in_eval"] & h["TVT_input"].isna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=["id", "tvt"])

    Xp = sub[FEATURE_COLS]
    delta_pred = np.mean([m.predict(Xp, num_iteration=m.best_iteration) for m in models], axis=0)
    sub["tvt"] = calibrate_well_tvt(sub, deltas_to_tvt(sub, delta_pred))
    sub["id"] = sub["well"] + "_" + sub["row_idx"].astype(str)
    return sub[["id", "tvt"]]


test_wells = list_wells(TEST_DIR)
print(f"Test wells: {len(test_wells)}")

pred_parts = [predict_eval_zone(w, TEST_DIR, models) for w in test_wells]
submission = pd.concat(pred_parts, ignore_index=True)

sample = pd.read_csv(SAMPLE_SUB)
submission = sample[["id"]].merge(submission, on="id", how="left")
missing = submission["tvt"].isna().sum()
if missing:
    raise ValueError(f"Missing predictions for {missing} sample ids")

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nWrote submission.csv  rows={len(submission):,}")
print(f"tvt range: {submission['tvt'].min():.2f} .. {submission['tvt'].max():.2f}")